# Notebook Overview
This is the second attempt to observe if accuracy improves when data is augmented. This notebook utilizes scaled imagegenerator operations to focus on the key areas of augmentation and the ones that take less computational power and resources . It trains on 1000 images

In [1]:
import tensorflow as tf

In [2]:
cifar = tf.keras.datasets.cifar10

In [3]:
(train_images,train_labels),(test_images,test_labels) = cifar.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step


In [4]:
#Normalizing dataset
train_images,test_images = train_images/255.0,test_images/255.0

In [5]:
'''
Imports the image module from Keras, which contains utility functions for image loading, saving, and processing.
'''

from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator #perfoms data augmentation

# creates a data generator object that transforms images
datagen = ImageDataGenerator(
rotation_range=40, # images will be randomly rotated by a degree of 40
width_shift_range=0.2,# Images will be randomly shifted horizontally by up to 20% of their total width.
height_shift_range=0.2,#img randomly shifted vertically
shear_range=0.2,# Images will be randomly sheared (tilted along an axis) by up to 20 degrees.
zoom_range=0.2, #Images will be randomly zoomed in by up to 20%
horizontal_flip=True, # randomly flipped
fill_mode='nearest')
'''
When transformations cause pixels to go outside the image boundaries (e.g., rotation, shift),
this specifies how to fill the newly created empty pixels (here, it fills them with the nearest pixel value).
'''

train_generator = datagen.flow(train_images,train_labels,batch_size =32)

In [7]:
shape = (32,32,3)

In [8]:
#Model Creation
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
model = tf.keras.Sequential()
Conv_layer1 = Conv2D(32, (3,3),activation ='relu',input_shape = shape) # First filter application
Max_pool1 = MaxPooling2D(2,2) # First pooled layer
Conv_layer2 = Conv2D(64, (3,3), activation = 'relu')#second filter application
Max_pool2 = MaxPooling2D(2,2) # Second pooled layer
Conv_layer3 = Conv2D(64,(3,3), activation = 'relu') #3rd filter application

#Adding Layers to sequential layer
model.add(Conv_layer1)
model.add(Max_pool1)
model.add(Conv_layer2)
model.add(Max_pool2)
model.add(Conv_layer3)
model.add(Flatten())
model.add(Dense(64, activation = 'relu'))
model.add(Dense(10, activation= 'softmax'))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,570 (478.79 KB)

 Trainable params: 122,570 (478.79 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
#compiling model
model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [13]:
#training image
model.fit(train_generator, epochs = 20, validation_data = (test_images,test_labels),steps_per_epoch= 2 * (len(train_images)//32))

Epoch 1/20
1562/3124 ━━━━━━━━━━━━━━━━━━━━ 51s 33ms/step - accuracy: 0.3243 - loss: 1.8143

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


3124/3124 ━━━━━━━━━━━━━━━━━━━━ 55s 17ms/step - accuracy: 0.3444 - loss: 1.7710 - val_accuracy: 0.4462 - val_loss: 1.5249
Epoch 2/20
3124/3124 ━━━━━━━━━━━━━━━━━━━━ 59s 19ms/step - accuracy: 0.4293 - loss: 1.5770 - val_accuracy: 0.5086 - val_loss: 1.3757
Epoch 3/20
3124/3124 ━━━━━━━━━━━━━━━━━━━━ 55s 18ms/step - accuracy: 0.4652 - loss: 1.4803 - val_accuracy: 0.5092 - val_loss: 1.3219
Epoch 4/20
3124/3124 ━━━━━━━━━━━━━━━━━━━━ 56s 18ms/step - accuracy: 0.4862 - loss: 1.4243 - val_accuracy: 0.5668 - val_loss: 1.2123
Epoch 5/20
3124/3124 ━━━━━━━━━━━━━━━━━━━━ 56s 18ms/step - accuracy: 0.5036 - loss: 1.3762 - val_accuracy: 0.5719 - val_loss: 1.2054
Epoch 6/20
3124/3124 ━━━━━━━━━━━━━━━━━━━━ 82s 18ms/step - accuracy: 0.5173 - loss: 1.3439 - val_accuracy: 0.5925 - val_loss: 1.1271
Epoch 7/20
3124/3124 ━━━━━━━━━━━━━━━━━━━━ 56s 18ms/step - accuracy: 0.5286 - loss: 1.3114 - val_accuracy: 0.5413 - val_loss: 1.3087
Epoch 8/20
3124/3124 ━━━━━━━━━━━━━━━━━━━━ 82s 18ms/step - accuracy: 0.5392 - loss: 1.28

# Training it on 50K Augmented Images to compare the non-augmented values and also same batches

In [14]:
#Model Creation
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
model = tf.keras.Sequential()
Conv_layer1 = Conv2D(32, (3,3),activation ='relu',input_shape = shape) # First filter application
Max_pool1 = MaxPooling2D(2,2) # First pooled layer
Conv_layer2 = Conv2D(64, (3,3), activation = 'relu')#second filter application
Max_pool2 = MaxPooling2D(2,2) # Second pooled layer
Conv_layer3 = Conv2D(64,(3,3), activation = 'relu') #3rd filter application

#Adding Layers to sequential layer
model.add(Conv_layer1)
model.add(Max_pool1)
model.add(Conv_layer2)
model.add(Max_pool2)
model.add(Conv_layer3)
model.add(Flatten())
model.add(Dense(64, activation = 'relu'))
model.add(Dense(10, activation= 'softmax'))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,570 (478.79 KB)

 Trainable params: 122,570 (478.79 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
#compiling model
model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [17]:
#training image
model.fit(train_generator, epochs = 20, validation_data = (test_images,test_labels),steps_per_epoch= 1563)

Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 57s 36ms/step - accuracy: 0.3567 - loss: 1.7425 - val_accuracy: 0.4934 - val_loss: 1.3874
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 55s 35ms/step - accuracy: 0.4297 - loss: 1.5696 - val_accuracy: 0.5159 - val_loss: 1.3490
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 55s 35ms/step - accuracy: 0.4658 - loss: 1.4788 - val_accuracy: 0.5231 - val_loss: 1.3640
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 55s 35ms/step - accuracy: 0.4920 - loss: 1.4083 - val_accuracy: 0.5544 - val_loss: 1.2433
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 56s 36ms/step - accuracy: 0.5122 - loss: 1.3496 - val_accuracy: 0.5901 - val_loss: 1.1421
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 81s 36ms/step - accuracy: 0.5261 - loss: 1.3103 - val_accuracy: 0.5884 - val_loss: 1.1647
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 56s 36ms/step - accuracy: 0.5412 - loss: 1.2869 - val_accuracy: 0.6148 - val_loss: 1.1075
Epoch 8/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 82s 35ms/step - accuracy: 0.5523 -